In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
notes_val = pd.read_csv("notes_val_predictions_task5.csv")
ecg_val = pd.read_csv("ecg_val_predictions_task5.csv")
structured_val = pd.read_csv("structured_val_predictions_task5.csv")

display(notes_val.head())
display(ecg_val.head())
display(structured_val.head())

print(len(notes_val))
print(len(ecg_val))
print(len(structured_val))

,sample_id,y_true,pred_prob_notes
0,0,1,0.977522
1,1,1,0.838746
2,2,1,0.680587
3,3,1,0.815637
4,4,1,0.895979


,sample_id,y_true,pred_prob_ecg
0,0,1,0.797777
1,1,1,0.410369
2,2,1,0.395739
3,3,1,0.700166
4,4,1,0.874622


,sample_id,y_true,pred_prob_structured
0,0,1,0.999043
1,1,1,0.998963
2,2,1,0.999180
3,3,1,0.895403
4,4,1,0.994455


18986
18986
18986


In [3]:
val_df = notes_val.merge(
    ecg_val[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_val[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)

val_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,1,0.977522,0.797777,0.999043
1,1,1,0.838746,0.410369,0.998963
2,2,1,0.680587,0.395739,0.999180
3,3,1,0.815637,0.700166,0.895403
4,4,1,0.895979,0.874622,0.994455
...,...,...,...,...,...
18981,18981,0,0.067917,0.487245,0.004924
18982,18982,0,0.778355,0.818380,0.012217
18983,18983,1,0.077378,0.588716,0.999176
18984,18984,1,0.901488,0.876360,0.999057


In [4]:
notes_test = pd.read_csv("notes_test_predictions_task5.csv")
ecg_test = pd.read_csv("ecg_test_predictions_task5.csv")
structured_test = pd.read_csv("structured_test_predictions_task5.csv")

display(notes_test.head())
display(ecg_test.head())
display(structured_test.head())

print(len(notes_test))
print(len(ecg_test))
print(len(structured_test))

,sample_id,y_true,pred_prob_notes
0,0,1,0.709161
1,1,0,0.967349
2,2,1,0.662742
3,3,1,0.973620
4,4,0,0.677587


,sample_id,y_true,pred_prob_ecg
0,0,1,0.801852
1,1,0,0.645319
2,2,1,0.588451
3,3,1,0.785665
4,4,0,0.735141


,sample_id,y_true,pred_prob_structured
0,0,1,0.997239
1,1,0,0.003398
2,2,1,0.999156
3,3,1,0.998999
4,4,0,0.004526


18986
18986
18986


In [5]:
test_df = notes_test.merge(
    ecg_test[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_test[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)
test_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,1,0.709161,0.801852,0.997239
1,1,0,0.967349,0.645319,0.003398
2,2,1,0.662742,0.588451,0.999156
3,3,1,0.973620,0.785665,0.998999
4,4,0,0.677587,0.735141,0.004526
...,...,...,...,...,...
18981,18981,1,0.799892,0.895833,0.999157
18982,18982,0,0.064653,0.743553,0.015759
18983,18983,0,0.683479,0.457049,0.009614
18984,18984,1,0.973418,0.616224,0.998945


In [6]:
# Feature engineering
def add_meta_features(df):
    eps = 1e-6
    
    # base probs
    s = df["pred_prob_structured"].astype(float)
    n = df["pred_prob_notes"].astype(float)
    e = df["pred_prob_ecg"].astype(float)
    
    # interaction terms
    df["notes_x_ecg"] = n * e
    df["notes_x_struct"] = n * s
    df["ecg_x_struct"] = e * s
    df["notes_x_ecg_x_struct"] = n * e * s
    
    # pairwise differences
    df["struct_minus_notes"] = s - n
    df["struct_minus_ecg"] = s - e
    df["notes_minus_ecg"] = n - e
    
    # summary stats across modalities
    df["prob_mean"] = (s + n + e) / 3.0
    df["prob_max"] = np.maximum(np.maximum(s, n), e)
    df["prob_min"] = np.minimum(np.minimum(s, n), e)
    df["prob_range"] = df["prob_max"] - df["prob_min"]
    
    # agreement / dispersion
    df["prob_std"] = np.std(np.vstack([s, n, e]), axis=0)
    
    # logit transform can help logistic stacking
    df["logit_struct"] = np.log((s + eps) / (1 - s + eps))
    df["logit_notes"] = np.log((n + eps) / (1 - n + eps))
    df["logit_ecg"] = np.log((e + eps) / (1 - e + eps))
    
    # rank-like indicators / thresholds
    df["struct_gt_notes"] = (s > n).astype(int)
    df["struct_gt_ecg"] = (s > e).astype(int)
    df["notes_gt_ecg"] = (n > e).astype(int)
    
    # confidence-style features
    df["struct_conf"] = np.abs(s - 0.5)
    df["notes_conf"] = np.abs(n - 0.5)
    df["ecg_conf"] = np.abs(e - 0.5)
    
    return df

val_df = add_meta_features(val_df.copy())
test_df = add_meta_features(test_df.copy())

# Features
stack_features = [
    # raw probs
    "pred_prob_structured",
    "pred_prob_notes",
    "pred_prob_ecg",
    
    # interactions
    "notes_x_ecg",
    "notes_x_struct",
    "ecg_x_struct",
    "notes_x_ecg_x_struct",
    
    # differences
    "struct_minus_notes",
    "struct_minus_ecg",
    "notes_minus_ecg",
    
    # summaries
    "prob_mean",
    "prob_max",
    "prob_min",
    "prob_range",
    "prob_std",
    
    # logits
    "logit_struct",
    "logit_notes",
    "logit_ecg",
    
    # comparisons
    "struct_gt_notes",
    "struct_gt_ecg",
    "notes_gt_ecg",
    
    # confidence
    "struct_conf",
    "notes_conf",
    "ecg_conf",
]

X_val = val_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val = val_df["y_true"].astype(int)

X_test = test_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["y_true"].astype(int)

# Logistic stacking model
meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

meta_model.fit(X_val, y_val)

# Predict + evaluate
test_df["pred_logistic_ensemble"] = meta_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_df["pred_logistic_ensemble"])
auprc = average_precision_score(y_test, test_df["pred_logistic_ensemble"])

print("Logistic multimodal test AUROC:", round(auc, 6))
print("Logistic multimodal test AUPRC:", round(auprc, 6))

# Inspect learned weights
lr = meta_model.named_steps["lr"]
coefs = pd.DataFrame({
    "feature": stack_features,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive coefficients:")
print(coefs.head(10).to_string(index=False))

print("\nTop negative coefficients:")
print(coefs.tail(10).to_string(index=False))

Logistic multimodal test AUROC: 0.997953
Logistic multimodal test AUPRC: 0.999073

Top positive coefficients:
             feature     coef
        logit_struct 5.502921
         notes_x_ecg 0.799747
         logit_notes 0.433218
            prob_std 0.245739
  struct_minus_notes 0.149501
        notes_gt_ecg 0.062703
    struct_minus_ecg 0.013544
pred_prob_structured 0.000000
            prob_max 0.000000
          prob_range 0.000000

Top negative coefficients:
        feature      coef
  struct_gt_ecg -0.020359
   ecg_x_struct -0.023314
struct_gt_notes -0.038711
       ecg_conf -0.059663
    struct_conf -0.062921
  pred_prob_ecg -0.082543
     notes_conf -0.091896
      logit_ecg -0.114140
pred_prob_notes -0.147319
 notes_x_struct -0.193879
